<a href="https://colab.research.google.com/github/tonygruyichich-glitch/Python_Workshop/blob/main/03_AI_Conversation_Analysis/Building_AI_Powered_Conversation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Building AI-Powered Conversation Analysis with Gemini & Python

Customer support conversations contain valuable business information, but most of it is stored as **unstructured text**.

Traditional data analysis works well when information is already organized into columns such as `topic`, `sentiment`, or `intent`. However, extracting these attributes from free-text conversations is difficult to do with simple rules and traditional Python code.

This is where an **LLM can add value**.

Instead of manually defining rules for every possible way a customer might express a problem, we can use Gemini to understand the conversation and extract meaningful information from the text.

### 🎯 Our Goal

Build a pipeline that transforms:

**Unstructured Conversation → Gemini → Structured Data → Pandas → Business Insights**

In this notebook, we will use Gemini to analyze customer conversations and extract information such as:

- **Topic**
- **Sentiment**
- **Urgency**

We will then use Python and Pandas to analyze the structured results.

# 📂 Load & Explore the Data

First, we will load the customer support conversations into a Pandas DataFrame.

Before using Gemini, we need to understand the structure of our data and identify the column that contains the conversation text.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/python_workshop/apple_support_new.csv"

df = pd.read_csv(file_path)

df.head()

In [ ]:
# 1. ביטול חיתוך טקסט בתוך התאים (מציג את אורך המלל המלא ללא ...)
pd.set_option('display.max_colwidth', None)

In [ ]:
df.head()

# Connect Gemini to Python

Now we will connect our Python notebook to **Gemini**.

The goal is to use the LLM directly from our Python code, rather than interacting with it through a chat interface.

First, we will make a simple request to the model to verify that the connection works.

In [ ]:
from google.colab import userdata
userdata.get('Gemini_API_Key')

In [ ]:
from google import genai
from google.colab import userdata

# 2. שליפת ה-API Key מתוך ה-Secrets של Colab
api_key = userdata.get('Gemini_API_Key')

# 3. אתחול ה-Client ושליחת בקשת בדיקה
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model='gemini-3.8-flash',
    contents="What is sentiment analysis?"
)

# 4. הדפסת התשובה
print(response.text)

### Understanding the Model Response

The model returns a **response object** that contains more than just the generated text.

The most important part for us is:

```python
response.text
```

This allows our Python code to access the model's answer and use it in the next steps of our analysis.

The response also contains useful metadata, such as:

- **Model version** – which model generated the response
- **Token counts** – how many tokens were used for the prompt and response
- **Total tokens** – the total amount of tokens used
- **Finish reason** – why the model stopped generating the response
- **Response ID** – a unique identifier for the response

This information can help us **monitor usage, estimate costs, debug our application, and analyze model performance**.

The key idea is: **we are not just chatting with an AI — we are integrating the AI into our Python code.**

In [ ]:
#what response contains
response

The important part is that Gemini is now being called **from Python code**.

Our Python program can:

1. Send information to the model
2. Receive the model's response
3. Use that response in the rest of our data pipeline

Next, instead of sending a general question, we will send Gemini an actual **customer conversation** from our dataset.

# Analyze a Single Comment

Now we will send an actual customer conversation to Gemini.

Instead of manually defining rules to identify what the customer wants, we will ask the LLM to understand the conversation and extract useful information from it.

For this first example, we will analyze:

- **Sentiment** — Is the customer positive, neutral, or negative?
- **Topic** — What is the conversation about?
- **Urgency** — How urgent is the issue?

In [ ]:
#a single comment
conversation = df["text"].iloc[0]

print(conversation)

#### Creating a Function to Communicate with the LLM

By creating a Python function that handles the communication with the LLM, we can make it easier to interact with the model inside loops.

Instead of writing the API call and handling the response each time, we can simply call the function whenever we need the LLM to analyze new data.

This makes our code **simpler, reusable, and easier to maintain**.

In [ ]:
def analyze_conversation_no_prompt(text):
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=text
    )

    return response.text

In [ ]:
result = analyze_conversation_no_prompt(df["text"].iloc[0])
print(result)

### ⚠️ Why Did We Get an Unexpected Answer?

When we sent the conversation to Gemini **without any instructions**, the model had to decide for itself what to do with the text.

As a result, it generated a long and unrelated response instead of extracting the information we need.

This is an important lesson when working with LLMs:

> **The model can understand the text, but it does not automatically know what we want to extract from it.**

We need to provide a **focused prompt** that clearly defines:

- What the model should analyze
- Which information we want
- What format the answer should have

This is especially important when the model's output will later be processed by Python.

### 💡 Focused Prompt

With a focused prompt, Gemini was able to extract meaningful information from the conversation and organize it into the categories we defined.

The model identified the customer's **sentiment, topic, intent, and urgency** based only on the conversation text.

This is much more useful than a general response because we are now asking the model to perform a **specific analytical task**.

However, the output is still written for a human to read. In the next step, we will ask Gemini to return the same information in a **structured JSON format**, so Python can work with it programmatically.

In [ ]:
def analyze_conversation_with_prompt(text):
    prompt = f"""
You are analyzing a customer support conversation.

Analyze the conversation and identify the following:

1. Sentiment:
   - Positive
   - Neutral
   - Negative

2. Topic:
   Identify the main subject of the conversation.

3. Urgency:
   - Low
   - Medium
   - High

Return a clear and concise analysis for each of the four categories.

Customer conversation:
{text}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text

In [ ]:
response.text

### Turning the Response into Structured Data

The previous response is useful for a human reader, but it is not easy for Python to analyze.

For example, we cannot easily create a column called `sentiment` from a paragraph of text.

Instead, we will ask Gemini to return the same analysis in a **structured JSON format**.

This will allow us to turn the model's response into Python data that can later become DataFrame columns.

In [ ]:
def analyze_conversation_structured_data(text):
    import json
    prompt = f"""
Analyze the following customer support conversation.

Extract these four attributes:

- sentiment: Positive, Neutral, or Negative
- topic: the main subject of the conversation
- intent: what the customer wants or needs
- urgency: Low, Medium, or High

Return ONLY a valid JSON object in exactly this format:

{{
    "sentiment": "...",
    "topic": "...",
    "urgency": "..."
}}

Do not add explanations, markdown, or any text outside the JSON.

Customer conversation:
{text}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    result_dict = json.loads(response.text)

    return result_dict

In [ ]:
#a test on a single comment
result = analyze_conversation_structured_data(df["text"].iloc[0])

print(result)

In [ ]:
result['topic']

# Scale the Analysis

So far, we have analyzed a single conversation.

The real value comes from applying the same process to **many conversations**.

We will use Python to:

1. Send each conversation to Gemini
2. Receive the structured response
3. Convert the responses into a new DataFrame

This turns our original unstructured text into a new **AI-generated analytical dataset**.

## Test on Three Texts
- Selected **3 conversations** from our dataset.
- Sent each conversation to the **LLM** for analysis.
- Received the results as **structured JSON**.
- Converted the JSON results into a **DataFrame**.
- Combined the AI results with the original conversation data.
- Now we have a dataset enriched with **AI-generated insights** that we can analyze with Pandas.

In [ ]:
# Create an empty list to store the results returned by the LLM
results = []

# Send the first 3 conversations to the LLM one by one
for text in df["text"].head(3):
    result = analyze_conversation_structured_data(text)
    results.append(result)

# Convert the JSON results returned by the LLM into a DataFrame
ai_results = pd.DataFrame(results)

# Reset the index of the original DataFrame
test_df = df.reset_index(drop=True)

# Combine the original conversation data with the AI-generated results
test_df = pd.concat(
    [test_df, ai_results],
    axis=1
)



In [ ]:
test_df.head(3)

### What did we achieve?

- We started with the original conversation data.
- We sent the conversations to the **LLM for analysis**.
- The LLM analyzed the text and returned structured information such as:
  - **Sentiment**
  - **Topic**
  - **Urgency**
- We added these AI-generated fields to our original DataFrame.
- We can now use **Pandas to analyze the conversations together with the insights generated by the LLM**.

##Covert all texts

### Batch Processing

Processing three conversations separately took about **40 seconds**, since each conversation required a separate API call.

To make the process faster, we switched to **Batch Processing**: we combine all conversations into a single request and send them to the model together.

In [ ]:
def analyze_conversation_structured_data_batch_processing(texts):
    import json

    conversations = "\n\n".join(
        [f"Conversation {i+1}:\n{text}" for i, text in enumerate(texts)]
    )

    prompt = f"""
Analyze the following customer support conversations.

For each conversation, extract:

- sentiment: Positive, Neutral, or Negative
- topic: only one of these: headphones, ipad, mac, iphone.
- urgency: Low, Medium, or High

Return a JSON array.

Each item must follow this format:

{{
    "sentiment": "...",
    "topic": "...",
    "urgency": "..."
}}

Return exactly one item for each conversation,
in the same order as the conversations below.

{conversations}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)

In [ ]:
results=[]
batch = df["text"].tolist()
batch_results = analyze_conversation_structured_data_batch_processing(batch)
results.extend(batch_results)

# Convert the JSON results returned by the LLM into a DataFrame
ai_results = pd.DataFrame(results)

# Reset the index of the original DataFrame
final_df = df.reset_index(drop=True)

# Combine the original conversation data with the AI-generated results
final_df = pd.concat(
    [final_df, ai_results],
    axis=1
)

In [ ]:
final_df

#Visualization Using Prompt

the good old way where we send the prompt to the outside LLM and recieving the code from it

In [ ]:
print('give me a grouped bar chart that shows first each topic sentiments counts')
print(final_df[['topic', 'sentiment','urgency']].to_string())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

sns.countplot(
    data=final_df,
    x="topic",
    hue="sentiment",
    order=["headphones", "ipad", "mac", "iphone"],
    hue_order=["Negative", "Neutral", "Positive"]
)

plt.title("Sentiment Distribution by Product", fontsize=16)
plt.xlabel("Product")
plt.ylabel("Number of Conversations")
plt.legend(title="Sentiment")

plt.tight_layout()
plt.show()

In [ ]:
print('give me a pie chart for each topic that shows the urgency distribution. use seaborn and matrix for the pies')
print(final_df[['topic', 'sentiment','urgency']].to_string())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

topics = final_df["topic"].unique()

# Create a 2x2 matrix of pie charts
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, topic in zip(axes.flat, topics):

    # Count urgency levels for the current topic
    urgency_counts = (
        final_df[final_df["topic"] == topic]["urgency"]
        .value_counts()
    )

    # Create the pie chart
    ax.pie(
        urgency_counts,
        labels=urgency_counts.index,
        autopct="%1.1f%%",
        startangle=90
    )

    ax.set_title(f"{topic.title()} - Urgency Distribution")

plt.suptitle(
    "Urgency Distribution by Topic",
    fontsize=16,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
print('do you have any suggestion for a good graph to draw this df')
print(final_df[['topic', 'sentiment','urgency']].to_string())

In [ ]:
sentiment_table

In [ ]:
sentiment_table = pd.crosstab(
    final_df["topic"],
    final_df["sentiment"]
)

plt.figure(figsize=(8, 5))

sns.heatmap(
    sentiment_table,
    annot=True,
    fmt="d",
    cmap="RdYlGn"
)

plt.title("Sentiment Distribution by Product")
plt.xlabel("Sentiment")
plt.ylabel("Product")

plt.tight_layout()
plt.show()

# Why Not Let the LLM Create the Visualization?

So far, we asked the LLM to generate Python code for the visualization we wanted.

But why stop there?

Instead of asking the model to **write the code and then running it ourselves**, we can build a Python function that sends the visualization request and the DataFrame directly to the LLM, receives the generated code, and executes it automatically.

This allows us to go from:

**Prompt → LLM → Python Code**

to:

**Prompt → LLM → Visualization 📊**

Let's build a function that does exactly that.

In [ ]:
def generate_visualization(prompt, df, n_rows=50):
    df = df[['tweet_id','topic', 'sentiment','urgency']]
    # Show only part of the DataFrame to the model
    data_sample = df.sample(n=n_rows, random_state=42).to_string()

    llm_prompt = f"""
You are a Python data visualization expert.

The user wants the following visualization:

{prompt}

Here is a sample of the DataFrame:

{data_sample}

Create Python code that generates exactly this visualization.

Requirements:
- Use pandas, seaborn and matplotlib when appropriate.
- The DataFrame is already available as `df`.
- Do not create fake data.
- Do not explain the code.
- Return ONLY executable Python code.
- The code must display the chart using plt.show().
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=llm_prompt
    )

    code = response.text

    # Remove Markdown code fences if the model added them
    code = code.replace("```python", "").replace("```", "").strip()

    # Execute the generated code
    exec(code, {
        "df": df,
        "pd": pd,
        "sns": sns,
        "plt": plt
    })

In [ ]:
generate_visualization(
    "Give me a grouped bar chart showing sentiment counts for each topic",
    final_df
)

In [ ]:
generate_visualization(
    "תן לי גרף עוגה שמראה את התפלגות הסנטימנט עבור כל המדגם",
    final_df
)

In [ ]:
generate_visualization(
    "תן לי מטריצה של גרפי עוגה אחד לכל topic שם אנו רואים את ההפלגות של הסנטימנטים",
    final_df
)

In [ ]:


generate_visualization(
    "do you have any suggestion for a good graph to draw this df",
    final_df
)

In [ ]:


generate_visualization(
    "do you have any suggestion for a good graph to draw this df",
    final_df
)